# KAMA – ajuster la reconnaissance vocale sur le pidgin camerounais (Kaggle)

Même entraînement que `entrainement_stt_colab.ipynb`, adapté à Kaggle. Données : Common Voice pidgin camerounais (et Nigerian Pidgin ASR s'il est dans le zip), notes avant/après sur dev, test Common Voice et SOREVA.

**Une seule fois :**
1. Compte Kaggle **vérifié par téléphone** (sinon pas de GPU ni d'Internet).
2. **Datasets → New Dataset** : envoyer `KAMA_pidgin_stt_drive.zip` (583 Mo), titre `kama-stt-wes`, visibilité **Private**. Données Common Voice : jamais public, ne pas partager le dataset. Kaggle peut décompresser le zip : le carnet accepte les deux formes.
3. Dans ce carnet : **Add Input** → ton dataset `kama-stt-wes`.
4. **Settings** (panneau de droite) : **Accelerator = GPU T4 x2** (pas P100), **Internet = On** (téléchargement du modèle de départ).

**Lancer :**
1. **Essai** : `ESSAI = True`, puis *Run All* en session interactive (quelques minutes : 8 enregistrements par jeu, 1 passe). Il vérifie que tout s'enchaîne jusqu'à la copie finale.
2. **Vrai entraînement** : `ESSAI = False`, puis **Save Version → Save & Run All (Commit)**. Le carnet tourne sur les serveurs de Kaggle, navigateur fermé. Durée attendue pour `wbert` : environ 2 à 4 h (15 passes) ; une session Kaggle est coupée à 12 h.
3. À la fin : onglet **Output** de la version → télécharger `stt_<modele>.zip` (modèle final + `resultats.json`).

Pour essayer un autre modèle, changer `MODELE` et relancer une nouvelle version.

## 1. Réglages et GPU

In [ ]:
import os, sys
import torch
assert torch.cuda.is_available(), 'Pas de GPU : Settings → Accelerator = GPU T4 x2'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # un seul T4 : réglages identiques à Colab (lot 8 × cumul 2)
print('Python', sys.version.split()[0], '|', torch.cuda.get_device_name(0))

ESSAI = True       # True : essai rapide (8 enregistrements par jeu, 1 passe). False : vrai entraînement.
MODELE = 'wbert'   # 'wbert' (conseillé), 'xlsr' (plus léger) ou 'mms' (adaptateur)

TMP = '/kaggle/temp' if os.path.isdir('/kaggle/temp') else '/tmp'   # disque de travail (non gardé)
D = f'{TMP}/KAMA/stt'
SORTIE = f'{TMP}/sortie_{MODELE}'
SORTIE_FINALE = '/kaggle/working'                                     # gardé dans l'onglet Output
print('mode', 'ESSAI' if ESSAI else 'ENTRAINEMENT COMPLET', '| modèle', MODELE, '| travail :', TMP)

## 2. Données
Cherche le zip dans les données ajoutées au carnet (`/kaggle/input`), ou sa version déjà décompressée par Kaggle, et le copie sur le disque de travail (toujours refait, pour ne jamais réutiliser d'anciennes données).

In [ ]:
import glob, shutil, zipfile
shutil.rmtree(f'{TMP}/KAMA', ignore_errors=True)
zips = glob.glob('/kaggle/input/**/KAMA_pidgin_stt_drive.zip', recursive=True)
if zips:
    print('zip :', zips[0])
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(TMP)
else:
    deja = glob.glob('/kaggle/input/**/stt/manifeste_cv_train.jsonl', recursive=True)
    assert deja, "Données introuvables : ajoute le dataset kama-stt-wes au carnet (Add Input)"
    src = os.path.dirname(deja[0])
    print('dossier déjà décompressé :', src)
    shutil.copytree(src, D)
for nom in ['cv_train', 'cv_dev', 'cv_test', 'soreva_test']:
    n = sum(1 for _ in open(f'{D}/manifeste_{nom}.jsonl', encoding='utf-8'))
    print(nom, n)
    assert n > 0
assert os.path.exists(f'{D}/entrainer_stt.py')

## 3. Bibliothèques
`transformers` reste en version 4 (la v5 a supprimé `warmup_ratio`, utilisé par le script). Nécessite **Internet = On**.

In [ ]:
!pip -q install jiwer soundfile accelerate 'transformers>=4.45,<5'
import transformers
print('transformers', transformers.__version__)
assert int(transformers.__version__.split('.')[0]) < 5, 'transformers 5 installé : Run → Restart & clear outputs, puis relancer'

## 4. Entraînement
L'essai passe `--limite 8 --epoques 1`. Les notes **AVANT** (modèle de départ) et **APRÈS** s'affichent dans la sortie : WER = mots faux, CER = lettres fausses (en %, plus bas = mieux).

In [ ]:
options = '--limite 8 --epoques 1' if ESSAI else ''
shutil.rmtree(SORTIE, ignore_errors=True)
!python {D}/entrainer_stt.py --donnees {D} --sortie {SORTIE} --modele {MODELE} {options}

## 5. Résultats et récupération
Seul le modèle final et `resultats.json` vont dans `/kaggle/working` (pas les points de reprise) : onglet **Output** → télécharger. Sur ton poste, le modèle va dans `KAMA/models/poids/stt/` (hors Git).

In [ ]:
import json
r = json.load(open(f'{SORTIE}/resultats.json'))
print('AVANT ', r.get('avant'))
print('APRES ', r['apres'])
suffixe = '_essai' if ESSAI else ''
dossier = f'{TMP}/stt_{MODELE}{suffixe}'
shutil.rmtree(dossier, ignore_errors=True)
shutil.copytree(f'{SORTIE}/final', f'{dossier}/final')
shutil.copy(f'{SORTIE}/resultats.json', dossier)
chemin = shutil.make_archive(f'{SORTIE_FINALE}/stt_{MODELE}{suffixe}', 'zip', root_dir=TMP, base_dir=os.path.basename(dossier))
print(chemin, round(os.path.getsize(chemin) / 1e6), 'Mo')